# How Special Are Legendary Pokémon? What Makes a Pokemon Legendary, and Can We Predict It?

**Author:** Maxi Hase
**Course:** DCS 211, Fall 2026

**Dataset:** "The Complete Pokemon Dataset" by Rounak Banik, found on Kaggle (https://www.kaggle.com/datasets/rounakbanik/pokemon)

This notebook is part of my final project for DCS 211, and seeks to explore whether there is any relationship between certain stats of a Pokemon and them being legendary, as well as whether we can predict whether a given Pokemon will be legendary.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay)


## A. Data Collection and Exploration

In [2]:
df = pd.read_csv('data/pokemon.csv') # saving the CSV file as a dataframe
df.shape # 801 rows, 41 columns
df.head()

,abilities,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,...,percentage_male,pokedex_number,sp_attack,sp_defense,speed,type1,type2,weight_kg,generation,is_legendary
0,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,1,65,65,45,grass,poison,6.9,1,0
1,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,2,80,80,60,grass,poison,13.0,1,0
2,"['Overgrow', 'Chlorophyll']",1.0,1.0,1.0,0.5,0.5,0.5,2.0,2.0,1.0,...,88.1,3,122,120,80,grass,poison,100.0,1,0
3,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,4,60,50,65,fire,NaN,8.5,1,0
4,"['Blaze', 'Solar Power']",0.5,1.0,1.0,1.0,0.5,1.0,0.5,1.0,1.0,...,88.1,5,80,65,80,fire,NaN,19.0,1,0


In [3]:
df.dtypes

abilities                str
against_bug          float64
against_dark         float64
against_dragon       float64
against_electric     float64
against_fairy        float64
against_fight        float64
against_fire         float64
against_flying       float64
against_ghost        float64
against_grass        float64
against_ground       float64
against_ice          float64
against_normal       float64
against_poison       float64
against_psychic      float64
against_rock         float64
against_steel        float64
against_water        float64
attack                 int64
base_egg_steps         int64
base_happiness         int64
base_total             int64
capture_rate             str
classfication            str
defense                int64
experience_growth      int64
height_m             float64
hp                     int64
japanese_name            str
name                     str
percentage_male      float64
pokedex_number         int64
sp_attack              int64
sp_defense    

In [4]:
df.describe()

,against_bug,against_dark,against_dragon,against_electric,against_fairy,against_fight,against_fire,against_flying,against_ghost,against_grass,...,height_m,hp,percentage_male,pokedex_number,sp_attack,sp_defense,speed,weight_kg,generation,is_legendary
count,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,801.000000,...,781.000000,801.000000,703.000000,801.000000,801.000000,801.000000,801.000000,781.000000,801.000000,801.000000
mean,0.996255,1.057116,0.968789,1.073970,1.068976,1.065543,1.135456,1.192884,0.985019,1.034020,...,1.163892,68.958801,55.155761,401.000000,71.305868,70.911361,66.334582,61.378105,3.690387,0.087391
std,0.597248,0.438142,0.353058,0.654962,0.522167,0.717251,0.691853,0.604488,0.558256,0.788896,...,1.080326,26.576015,20.261623,231.373075,32.353826,27.942501,28.907662,109.354766,1.930420,0.282583
min,0.250000,0.250000,0.000000,0.000000,0.250000,0.000000,0.250000,0.250000,0.000000,0.250000,...,0.100000,1.000000,0.000000,1.000000,10.000000,20.000000,5.000000,0.100000,1.000000,0.000000
25%,0.500000,1.000000,1.000000,0.500000,1.000000,0.500000,0.500000,1.000000,1.000000,0.500000,...,0.600000,50.000000,50.000000,201.000000,45.000000,50.000000,45.000000,9.000000,2.000000,0.000000
50%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,65.000000,50.000000,401.000000,65.000000,66.000000,65.000000,27.300000,4.000000,0.000000
75%,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,2.000000,1.000000,1.000000,1.000000,...,1.500000,80.000000,50.000000,601.000000,91.000000,90.000000,85.000000,64.800000,5.000000,0.000000
max,4.000000,4.000000,2.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,14.500000,255.000000,100.000000,801.000000,194.000000,230.000000,180.000000,999.900000,7.000000,1.000000


In [5]:
df['is_legendary'].value_counts() # 0: 731, 1: 70

is_legendary
0    731
1     70
Name: count, dtype: int64

In [6]:
# this first checks each entry in the dataframe for whether 
# it has a missing value (NaN), then sums up the number of missing values per column, and finally
# sorts the columns by how many missing values exist
df.isna().sum().sort_values(ascending=False) 

type2                384
percentage_male       98
weight_kg             20
height_m              20
name                   0
capture_rate           0
classfication          0
defense                0
experience_growth      0
hp                     0
japanese_name          0
abilities              0
base_happiness         0
pokedex_number         0
sp_attack              0
sp_defense             0
speed                  0
type1                  0
generation             0
base_total             0
base_egg_steps         0
against_bug            0
attack                 0
against_dark           0
against_dragon         0
against_electric       0
against_fairy          0
against_fight          0
against_fire           0
against_flying         0
against_ghost          0
against_grass          0
against_ground         0
against_ice            0
against_normal         0
against_poison         0
against_psychic        0
against_rock           0
against_steel          0
against_water          0


## Summary of Section A:

**Size and Structure:** The dataset contains 801 rows along with 41 features. Each row represents one Pokémon, and the features include abilities, stats (e.g. Attack, Defense, etc.), type, capture rate, among others.

**Data Types:** Most features are numerical values, as would be expected. There are also some expected string features, such as the name, or the type. However, there is one string feature that should be numerical: the capture rate. This suggests that there is at least one Pokémon with a non-numerical capture rate, which can either be a mistake in the dataset, or a quirk that a few Pokémon have. This will be investigated in Section B.

**Distribution of Legendary Pokémon:** There are 70 legendary Pokémon, and 731 non-legendary Pokémon. This makes about 8.7% of Pokémon in this Dataset legendary. This means two things for our context: For one, there is limited data to train on and to see patterns in, and secondly, the data is heavily skewed towards non-legendary Pokémon. If a model simply guessed that every given Pokémon is not legendary, it would be right 91.3% of the time, so this is the baseline that this prediction model would have to beat.

**Missing values:** There are four features with missing entries for some Pokémon. The two most frequent are to be expected: most notably, 384 Pokémon are missing an entry for a second type. This is most likely because these Pokémon only have one type, so there is no second type to list. For example, while Venusaur is of Type Grass and Poison, Turtoise only has Type Water. Additionally, there are 98 Pokémon without a percentage of male Pokémon (to female), which is also not quite as surprising, because some Pokémon don't have a gender, such as Porygon. 
The other two features are more curious, however. Namely, there are 20 missing entries both for height and weight. These will need to be investigated in the next section.

## B. Data Cleansing and Preprocessing

In [7]:
# creating a working copy of the dataframe
df_clean = df.copy()

In [8]:
# checking any duplicate entries
print(df_clean.duplicated().sum()) # number of identical rows: 0
print(df_clean['name'].duplicated().sum()) # number of identical names: 0

0
0


In [9]:
# inspecting the invalid values for the capture rate
converted = pd.to_numeric(df_clean['capture_rate'], errors='coerce') # turn entries into numerical values
df_clean.loc[converted.isna(), ['name', 'capture_rate']] # show all names and capture rates for Pokémon where conversion failed

,name,capture_rate
773,Minior,30 (Meteorite)255 (Core)


In [10]:
# the meteorite form is the default form, so we will use that form
# replacing the entry with capture rate 30
df_clean.loc[df_clean['name']=='Minior', 'capture_rate'] = '30'
#turning all entries into numerical entries
df_clean['capture_rate'] = pd.to_numeric(df_clean['capture_rate'])

# print data type to check
print(df_clean['capture_rate'].dtype) # int64

int64


In [11]:
print(df_clean.loc[df_clean['name']=='Minior',['name','capture_rate']])

       name  capture_rate
773  Minior            30


In [12]:
# checking which Pokémon have missing height entries
df_clean.loc[df_clean['height_m'].isna(), ['name', 'generation', 'is_legendary']]

,name,generation,is_legendary
18,Rattata,1,0
19,Raticate,1,0
25,Raichu,1,0
26,Sandshrew,1,0
27,Sandslash,1,0
36,Vulpix,1,0
37,Ninetales,1,0
49,Diglett,1,0
50,Dugtrio,1,0
51,Meowth,1,0


In [13]:
# checking which Pokémon have missing weight entries
df_clean.loc[df_clean['weight_kg'].isna(), ['name', 'generation', 'is_legendary']]
# Note that these are the same Pokémon that are missing height entries
# all of these Pokémon have alternate forms

,name,generation,is_legendary
18,Rattata,1,0
19,Raticate,1,0
25,Raichu,1,0
26,Sandshrew,1,0
27,Sandslash,1,0
36,Vulpix,1,0
37,Ninetales,1,0
49,Diglett,1,0
50,Dugtrio,1,0
51,Meowth,1,0


In [14]:
#checking if there are other Pokémon that use different forms
df_clean.loc[df_clean['type1']==df_clean['type2'],['name','type1','type2']]
# all of these Pokémon have alternate forms, and here again their types get confused. Perhaps this is the case for all Pokémon with
# alternate forms?

,name,type1,type2
25,Raichu,electric,electric
49,Diglett,ground,ground
50,Dugtrio,ground,ground
87,Grimer,poison,poison
88,Muk,poison,poison
491,Shaymin,grass,grass
554,Darmanitan,fire,fire


In [15]:
# checking stats for some Mega-Pokemon, as these have inflated stats, and would most affect the results:
print(df_clean.loc[df_clean['name']=='Charizard',['base_total']])
print(df_clean.loc[df_clean['name']=='Tyranitar',['base_total']])
print(df_clean.loc[df_clean['name']=='Rayquaza',['base_total']])


   base_total
5         634
     base_total
247         700
     base_total
383         780


### Alternate Forms in the Data

Investigating the discrepancies further and trying to fix them has revealed some bigger issues. At first, it seemed that there was a problem only with Alolan form Pokémon that were missing entries for their height and weight. Using the Alolan forms instead of the default Kantonian forms does not affect the stats in a noticeable manner, so I decided to simply use those forms at first.

Further investigations revealed, however, that many Pokémon with alternate forms, use the alternate stats and entries, with some mixed types. While the mixed types would most likely not affect the results by a lot, some of the alternate stats massively impact the analysis. Most notably, Pokémon with mega evolutions receive a total stat boost of 100, which pushes many non-legendary Pokémon to be stronger than legendary Pokémon. This would most likely lead to a lot of non-legendary Pokémon to be identified as legendary. This is a big discovery, and needs fixing.

To address this issue, I will use a more reliable API that lists all default stats. In a different script, I will convert this API to a CSV file, that I will use to finish cleaning up the data here. Here is the link to the home page of the API: https://pokeapi.co/

The source is PokéAPI, with this documentation: https://pokeapi.co/v2/openapi/operations/pokemon_retrieve
The whole address for each Pokémon is https://pokeapi.co/api/v2/pokemon/{pokedex_number}, where {pokedex_number} is to be replaced with a Pokémon's pokedex number. It then returns a JSON object describing one Pokémon's default form. I will focus on stats, types, height and weight, as the other entries are not relevant to this data analysis.

In [16]:
# loading and inspecting the API dataset
api_df = pd.read_csv('data/pokeapi.csv')
print(api_df.shape)
print(api_df.isna().sum())

api_df.head()

(801, 12)
pokedex_number      0
api_name            0
hp                  0
attack              0
defense             0
sp_attack           0
sp_defense          0
speed               0
height_m            0
weight_kg           0
type2             400
type1               0
dtype: int64


,pokedex_number,api_name,hp,attack,defense,sp_attack,sp_defense,speed,height_m,weight_kg,type2,type1
0,1,bulbasaur,45,49,49,65,65,45,0.7,6.9,poison,grass
1,2,ivysaur,60,62,63,80,80,60,1.0,13.0,poison,grass
2,3,venusaur,80,82,83,100,100,80,2.0,100.0,poison,grass
3,4,charmander,39,52,43,60,50,65,0.6,8.5,NaN,fire
4,5,charmeleon,58,64,58,80,65,80,1.1,19.0,NaN,fire


In [20]:
# checking if all pokedex numbers line up
print((df_clean['pokedex_number']==api_df['pokedex_number']).all()) # all() is True only if all pokedex entries line up
# investigating which Pokémon have stat changes or type changes
any_changed = ((abs(df_clean['hp'] - api_df['hp'])
            + abs(df_clean['attack']-api_df['attack'])
            + abs(df_clean['defense']-api_df['defense'])
            + abs(df_clean['sp_attack']-api_df['sp_attack'])
            + abs(df_clean['sp_defense']-api_df['sp_defense'])
            + abs(df_clean['speed']-api_df['speed']) != 0)
            |(df_clean['type1'] != api_df['type1']) | (df_clean['type2'].fillna('none') != api_df['type2'].fillna('none')))
# making sure all columns are shown after this
pd.set_option('display.max_rows', 100)
print(any_changed.sum())
df_clean.loc[any_changed, ['name', 'hp', 'attack', 'defense', 'sp_attack',
                           'sp_defense', 'speed', 'type1', 'type2']]

True
85


,name,hp,attack,defense,sp_attack,sp_defense,speed,type1,type2
2,Venusaur,80,100,123,122,120,80,grass,poison
5,Charizard,78,104,78,159,115,100,fire,flying
8,Blastoise,79,103,120,135,115,78,water,NaN
14,Beedrill,65,150,40,15,80,145,bug,poison
17,Pidgeot,83,80,80,135,80,121,normal,flying
18,Rattata,30,56,35,25,35,72,normal,dark
19,Raticate,75,71,70,40,80,77,normal,dark
25,Raichu,60,85,50,95,85,110,electric,electric
26,Sandshrew,50,75,90,10,35,40,ground,ice
27,Sandslash,75,100,120,25,65,65,ground,ice


In [ ]:
# all the above Pokémon have some alternate form, except for Cresselia
# creating a feature of the sum of stats for the API dataframe
api_df['base_total'] = api_df['hp'] + api_df['attack'] + api_df['defense'] + api_df['sp_attack'] + api_df['sp_defense'] + api_df['speed']

In [19]:
# checking which Pokémon have no stat changes, but changes otherwise (e.g. types)
total_unchanged = df_clean['base_total'] == api_df['base_total']
df_clean.loc[any_changed & total_unchanged, ['name', 'type1', 'type2']]

,name,type1,type2
18,Rattata,normal,dark
19,Raticate,normal,dark
25,Raichu,electric,electric
26,Sandshrew,ground,ice
27,Sandslash,ground,ice
36,Vulpix,fire,ice
37,Ninetales,fire,ice
49,Diglett,ground,ground
50,Dugtrio,ground,ground
51,Meowth,normal,dark


In [25]:
# Coming back to Cresselia, Cresselia's stats were changed in a later game (https://bulbapedia.bulbagarden.net/wiki/Cresselia_(Pok%C3%A9mon))
# Instead, I will use the original dataset's stats for Cresselia

stat_columns = ['base_total', 'hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed']
api_df.loc[api_df['api_name'] == 'cresselia', stat_columns] = df.loc[df['name'] == 'Cresselia', stat_columns].values
print(api_df.loc[api_df['api_name']=='cresselia', ['base_total']])

     base_total
487         600


In [27]:
# finally, I will replace all the original dataset's stat columns with the API stats
replace_columns = ['hp', 'attack', 'defense', 'sp_attack', 'sp_defense', 'speed', 
                   'type1', 'type2', 'height_m', 'weight_kg', 'base_total']
df_clean[replace_columns] = api_df[replace_columns]

In [ ]:
# verifying
print(df_clean[['height_m', 'weight_kg']].isna().sum()) # 0 expected
print((df_clean['type1']==df_clean['type2']).sum()) # 0 expected
df_clean.loc[df_clean['name'].isin(['Charizard', 'Tyranitar', 'Cresselia']), ['name', 'base_total']] #small sample to check

height_m     0
weight_kg    0
dtype: int64
0


,name,base_total
5,Charizard,534
247,Tyranitar,600
487,Cresselia,600


The above verification shows that the stats, types, and height and weight follow the default forms now, so we can move on to the 'percentage_male' feature.

In [26]:
# next, I will investigate and clean the percentage_male feature
df_clean['percentage_male'].value_counts()

percentage_male
50.0     501
88.1     111
0.0       27
24.6      24
100.0     19
75.4      19
11.2       2
Name: count, dtype: int64